# 1. La cible : buts par équipe et total de buts

**Question** : comment se distribue le nombre de buts, selon le championnat et la saison, et une loi de Poisson le décrit-elle bien ?

> Notebook **descriptif** (partie 3, ADR-0034). Il ne sert ni à choisir une variable ni à régler un paramètre :
> la sélection appartient au protocole de la partie 4 (validation glissante, ablation). Corrélation n'est pas apport.
> Données : période de développement seulement (matchs antérieurs au 1er juillet 2025, scellé de l'ADR-0012),
> lues par la porte `foot_predictor.features.sources`. Sorties effacées au commit (`nbstripout`) : réexécuter pour les voir.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Seule porte vers les données (ADR-0028) : jeu versionné et, au besoin, matchs de la période de développement.
from foot_predictor.features.sources import load_dataset

pd.set_option("display.width", 140)
NAMES = {
    39: "PL",
    40: "Champ.",
    140: "Liga",
    141: "Segunda",
    78: "BL",
    79: "2. BL",
    135: "Serie A",
    136: "Serie B",
    61: "L1",
    62: "L2",
}
TOP5 = [39, 140, 78, 135, 61]
frame, manifest = load_dataset()
print("Jeu de données :", manifest["version"], "-", len(frame), "lignes, jusqu'au", frame["match_day"].max())
home = frame[frame["is_home"]].copy()  # une ligne par match (celle de l'équipe à domicile)
home["total"] = home["goals_for"] + home["goals_against"]

## Total de buts par match, par championnat et par saison

In [ ]:
total = home.pivot_table(index="season_year", columns="api_league_id", values="total", aggfunc="mean")
total = total.rename(columns=NAMES)
display(total.round(2))
ax = total[[NAMES[c] for c in TOP5]].plot(figsize=(9, 4), marker="o")
ax.set_title("Buts par match, top 5")
ax.set_xlabel("Saison (année de début)")
ax.set_ylabel("Buts par match")
plt.show()

## Avantage du terrain dans le temps (top 5)

In [ ]:
top = home[home["api_league_id"].isin(TOP5)]
adv = top.groupby("season_year").agg(
    ecart_buts=("goals_for", lambda s: (s - top.loc[s.index, "goals_against"]).mean()),
    victoires_domicile=("goals_for", lambda s: (s > top.loc[s.index, "goals_against"]).mean()),
    matchs=("goals_for", "size"),
)
display(adv.round(3))
ax = adv["ecart_buts"].plot(figsize=(9, 3.5), marker="o")
ax.axhline(0, color="grey", lw=0.8)
ax.set_title("Buts domicile − buts extérieur, par match (top 5)")
plt.show()

## Adéquation d'une loi de Poisson

Pour une loi de Poisson, la variance égale la moyenne : on compare le rapport variance / moyenne, par équipe et pour le total.

In [ ]:
rows = []
for league, group in frame.groupby("api_league_id"):
    team = group["goals_for"]
    tot = home.loc[home["api_league_id"] == league, "total"]
    rows.append(
        {
            "championnat": NAMES[league],
            "moyenne équipe": team.mean(),
            "variance équipe": team.var(),
            "rapport équipe": team.var() / team.mean(),
            "moyenne total": tot.mean(),
            "variance total": tot.var(),
            "rapport total": tot.var() / tot.mean(),
        }
    )
display(pd.DataFrame(rows).round(3))

In [ ]:
from math import exp, factorial

tot = home.loc[home["api_league_id"].isin(TOP5), "total"]
lam = tot.mean()
k = np.arange(0, 10)
observed = np.array([(tot == i).mean() for i in k])
poisson = np.array([exp(-lam) * lam**i / factorial(int(i)) for i in k])
display(pd.DataFrame({"buts": k, "observé": observed, "Poisson(moyenne)": poisson}).round(4))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(k - 0.2, observed, width=0.4, label="observé")
ax.bar(k + 0.2, poisson, width=0.4, label=f"Poisson({lam:.2f})")
ax.set_xlabel("Total de buts")
ax.set_ylabel("Fréquence")
ax.legend()
ax.set_title("Total de buts, top 5, 2000-01 à 2024-25")
plt.show()

## Matchs à huis clos (2019-20 à 2020-21)

Comparaison, sur les saisons 2019 et 2020 seulement, des matchs en période sûre de huis clos et des autres.

In [ ]:
period = home[home["season_year"].isin([2019, 2020])]
hc = period.groupby("behind_closed_doors").agg(
    matchs=("total", "size"),
    total=("total", "mean"),
    ecart_domicile=("goals_for", lambda s: (s - period.loc[s.index, "goals_against"]).mean()),
    victoires_domicile=("goals_for", lambda s: (s > period.loc[s.index, "goals_against"]).mean()),
)
display(hc.round(3))

## Conclusion

- **Niveau** : le total moyen va de 2,0 à 3,3 buts selon le championnat et la saison. La Bundesliga est la plus prolifique (2,95 buts par match sur la période), les D2 française, italienne et espagnole les moins (2,3 à 2,4). Hausse récente en Premier League (3,28 en 2023-24) et en Ligue 1 (2,98 en 2024-25) : l'effet saison ne se résume pas à une constante.
- **Avantage du terrain** : en baisse sur 25 ans dans le top 5, de +0,51 but par match en 2000-01 à +0,17 en 2024-25 ; la part des victoires à domicile passe de 50 % à 42 %. Creux en 2020-21 (+0,15), la saison des huis clos.
- **Poisson** : le rapport variance / moyenne du total est compris entre 0,99 et 1,06 selon le championnat, donc proche de 1 : en marginal, une loi de Poisson décrit bien le total. Par équipe, il va de 0,97 à 1,14 : une légère surdispersion, attendue quand on mélange des équipes de forces différentes. La dispersion *conditionnelle* aux variables reste à tester en partie 4 (étape M4). Le 0-0 est un peu plus fréquent qu'une Poisson ne le prévoit (7,7 % contre 6,8 %), le total de 1 un peu moins : c'est la signature connue de la dépendance entre les deux buts (Dixon-Coles, étape M5).
- **Huis clos** (saisons 2019-20 et 2020-21) : écart domicile − extérieur de 0,19 but à huis clos contre 0,29 hors huis clos, victoires à domicile 40,6 % contre 43,2 %, total presque inchangé (2,60 contre 2,62). C'est une description : l'effet se mêle à la tendance de fond, et l'apport de l'indicateur se mesure en partie 4.